In [0]:
%pip install git+https://github.com/jawong3/Databricks_Demos_and_Utilities.git@main
%restart_python

In [0]:
from pyspark.sql import functions as f
from dbx_utils.feature_flag import *

dbutils.widgets.dropdown("NEW_CASTING", "True", ["True", "False"], "Enable Replacement Feature")
dbutils.widgets.dropdown("ADDITIONAL_GOVERANCE", "True", ["True", "False"], "Enable New Feature")

FEDERAL_BRACKETS_2026 = [(58_523, "14%"), (117_045, "20.5%"), (181_440, "26%"), (258_482, "29%")]
TOP_RATE = "33%"

df = spark.createDataFrame(
    [
        (1, "Alice", "Toronto", 34, 85000.0, "T2024-01-15"),
        (2, "Bob", "Montreal", 28, 62000.0, "T2024-02-03"),
        (3, "Carol", "Vancouver", 45, 120000.0, "T2023-11-20"),
        (3, "Carol", "Vancouver", 45, 120000.0, "T2023-11-20"),
        (4, "Dave", "Calgary", 39, 74500.0, "T2024-03-09"),
        (5, "Eve", "Ottawa", 31, 91000.0, "T2024-04-22")],
    schema="id INT, name STRING, city STRING, age INT, salary DOUBLE, hire_date STRING")

df.show()

In [0]:
NEW_CASTING = dbutils.widgets.get("NEW_CASTING") == "True"
ADDITIONAL_GOVERANCE = dbutils.widgets.get("ADDITIONAL_GOVERANCE") == "True"

def timestamp_cast(df):
    return (df
        .withColumn("hire_date", f.replace(f.col("hire_date"), f.lit("T"), f.lit("")).cast("timestamp"))
        .select("*"))

@feature_flag(NEW_CASTING, fallback=timestamp_cast)
def date_cast(df):
    return (df
        .withColumn("hire_date", f.replace(f.col("hire_date"), f.lit("T"), f.lit("")).cast("date"))
        .select("*"))

def salary_pseudonymization(df):
    col = f.col("salary")
    expr = f.when(col.isNull(), f.lit(None))
    for upper, rate in FEDERAL_BRACKETS_2026:
        expr = expr.when(col <= upper, f.lit(rate))
    return (df
        .withColumn("tax_bracket", expr.otherwise(f.lit(TOP_RATE)))
        .drop("salary"))

@feature_flag(ADDITIONAL_GOVERANCE)
def age_pseudonymization(df):
    lo = f.floor(f.col("age") / 10) * 10
    return (df
        .withColumn("age_bracket", f.format_string("[%d-%d]", lo, lo + 10))
        .drop("age"))

In [0]:
# Toggle the variables on the parameter panel and run the cell again
# to see the effect of the feature flags
final_df = (df
    .transform(salary_pseudonymization)
    .transform(age_pseudonymization)
    .transform(date_cast)
)

final_df.show()